<a href="https://colab.research.google.com/github/achuthb-reizend/Shift/blob/main/Shift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ortools gspread

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 wh

RUN THIS FOR AUTHENTICATING WITH GOOGLE SHEETS

In [ ]:
import gspread
from google.colab import auth
from google.auth import default

def authenticate_sheets():
    print("Requesting Google Auth...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("Authentication successful! You are connected to Google Sheets.")
    return gc

# Run the auth and store the connection in 'gc'
gc = authenticate_sheets()

Requesting Google Auth...
Authentication successful! You are connected to Google Sheets.


RUN THIS TO GENERATE THE SHEET FOR THE MONTH


In [ ]:
# ==========================================
# --- CONFIGURATION & EMPLOYEE SETUP ---
# ==========================================

# 1. Date Setup
TARGET_MONTH = 3 # e.g., 4 for April
TARGET_YEAR = 2026

# 2. Public Holidays (Dates of the month)
PUBLIC_HOLIDAYS = []

# 3. Employee Rules (Achu = 6 days max, Full Time = 5 days max)
EMPLOYEES = {
    'Achu':     {'id': 'A', 'role': 'System_Engg', 'shifts': [0, 1, 2], 'color': {"red": 0.8, "green": 0.9, "blue": 1.0}},
    'Sidharth': {'id': 'B', 'role': 'Full_Time',   'shifts': [0, 1, 2], 'color': {"red": 0.9, "green": 0.8, "blue": 1.0}},
    'Rinshad':  {'id': 'C', 'role': 'Full_Time',   'shifts': [0, 2],    'color': {"red": 0.8, "green": 1.0, "blue": 0.8}},
    'Sanjeev':  {'id': 'D', 'role': 'Full_Time',   'shifts': [0, 1, 2], 'color': {"red": 1.0, "green": 0.9, "blue": 0.7}},
    'Sinto':    {'id': 'E', 'role': 'Full_Time',   'shifts': [0],       'color': {"red": 1.0, "green": 0.8, "blue": 0.8}}
}

# 4. AI Scoring Weights
WEIGHTS = {
    'MAIN_WORK_DAY': 200,
    'STAY_SAME_SHIFT': 150,
    'DOUBLE_OFF_REWARD': 100,
    'SINTO_WORK_ADJ': -50,
    'SINGLE_OFF_PENALTY': -75
}

# ---------------------------------------------------------
# --- BACKGROUND LOGIC (Do not edit below this line) ---
# ---------------------------------------------------------
SHEET_NAME = 'Shift_Planner'
INPUT_TAB = f"Input_{TARGET_MONTH}_{TARGET_YEAR}"
SCHED_TAB = f"Schedule_{TARGET_MONTH}_{TARGET_YEAR}"

if TARGET_MONTH == 1:
    PREV_MONTH, PREV_YEAR = 12, TARGET_YEAR - 1
else:
    PREV_MONTH, PREV_YEAR = TARGET_MONTH - 1, TARGET_YEAR
PREV_SCHED_TAB = f"Schedule_{PREV_MONTH}_{PREV_YEAR}"

emp_map = {name: data['id'] for name, data in EMPLOYEES.items()}
emp_ids = list(emp_map.values())
reverse_map = {v: k for k, v in emp_map.items()}
COLOR_MAP = {name: data['color'] for name, data in EMPLOYEES.items()}

print(f"✅ Configuration loaded for {TARGET_MONTH}/{TARGET_YEAR}. Ready to run the AI engine!")

✅ Configuration loaded for 3/2026. Ready to run the AI engine!


In [ ]:
import calendar
from datetime import datetime

# Define month and year variables here

# Open the spreadsheet using the authenticated gc object
# The sheet name 'Shift_Planner' is hardcoded here, assuming it's a fixed name.
# If the sheet name is dynamic, ensure SHEET_NAME is defined earlier.
sh = gc.open('Shift_Planner') # Assuming 'Shift_Planner' is the main spreadsheet name

def create_monthly_planner():

    # Validate inputs
    if not (1 <= month <= 12):
        print("Invalid month. Month must be between 1 and 12.")
        return

    sheet_name = f"Input_{month}_{year}"

    # 2. Check if sheet exists, or create it
    try:
        input_ws = sh.worksheet(sheet_name)
        input_ws.clear()
        print(f"Clearing existing sheet: {sheet_name}")
    except:
        input_ws = sh.add_worksheet(title=sheet_name, rows="100", cols="10")
        print(f"Creating new sheet: {sheet_name}")

    # 3. Headers
    employees = ["Achu", "Sidharth", "Rinshad", "Sanjeev", "Sinto"]
    headers = ["Date"] + employees

    # 4. Generate Dates
    num_days = calendar.monthrange(year, month)[1]
    rows = [headers]  # First row is headers

    for day in range(1, num_days + 1):
        # Format date as 'Mon 1', 'Tue 2', etc.
        date_obj = datetime(year, month, day)
        day_label = date_obj.strftime("%a %d")
        rows.append([day_label])  # Dates go in column A, leave others blank

    # 5. Push to Google Sheets
    input_ws.update(values=rows, range_name='A1')

    # 6. Formatting (Colors & Bold)
    # Header: Green background, bold
    # Date Column: Light grey, bold
    requests = [
        # Header Row Formatting
        {
            "repeatCell": {
                "range": {"sheetId": input_ws.id, "startRowIndex": 0, "endRowIndex": 1, "startColumnIndex": 0, "endColumnIndex": 6},
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": {"red": 0.85, "green": 0.92, "blue": 0.83},
                        "textFormat": {"bold": True}
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,textFormat)"
            }
        },
        # Date Column Formatting
        {
            "repeatCell": {
                "range": {"sheetId": input_ws.id, "startRowIndex": 1, "endRowIndex": num_days + 1, "startColumnIndex": 0, "endColumnIndex": 1},
                "cell": {
                    "userEnteredFormat": {
                        "backgroundColor": {"red": 0.95, "green": 0.95, "blue": 0.95},
                        "textFormat": {"bold": True}
                    }
                },
                "fields": "userEnteredFormat(backgroundColor,textFormat)"
            }
        },
        # Freeze the top row
        {
            "updateSheetProperties": {
                "properties": {"sheetId": input_ws.id, "gridProperties": {"frozenRowCount": 1}},
                "fields": "gridProperties.frozenRowCount"
            }
        }
    ]

    sh.batch_update({'requests': requests})
    print(f"Successfully generated {sheet_name}!")

# Call the function
if __name__ == "__main__":
    create_monthly_planner()

# To use different month/year, just change the variables at the top of the file
# Example:
# month = 3  # March
# year = 2026

Clearing existing sheet: Input_3_2026
Successfully generated Input_3_2026!


AI FOR SHIFT GENERATION

In [ ]:
from ortools.sat.python import cp_model
import math
import calendar
from datetime import datetime
import gspread

print("🚀 Starting AI Scheduling Engine...")

# ==========================================
# --- 1. DYNAMIC SHEET ACCESS ---
# ==========================================
print(f"📂 Connecting to Google Sheets (Target: {TARGET_MONTH}/{TARGET_YEAR})...")
sh = gc.open(SHEET_NAME)
input_ws = sh.worksheet(INPUT_TAB)

try:
    sched_ws = sh.worksheet(SCHED_TAB)
except gspread.exceptions.WorksheetNotFound:
    sched_ws = sh.add_worksheet(title=SCHED_TAB, rows="100", cols="10")

# ==========================================
# --- 2. HISTORY FETCHING (COMMA-AWARE) ---
# ==========================================
print("⏪ Fetching historical schedule data...")
def get_history():
    try:
        prev_ws = sh.worksheet(PREV_SCHED_TAB)
        data = prev_ws.get_all_records()
        last_week = data[-7:]
        hist = []
        for day in last_week:
            day_map = {eid: 'L' for eid in emp_ids}
            m_str = str(day.get('7am to 4pm', ''))
            e_str = str(day.get('3pm to 12am', ''))
            n_str = str(day.get('11pm to 8am', ''))

            for name, eid in emp_map.items():
                if name in m_str: day_map[eid] = 'M'
                elif name in e_str: day_map[eid] = 'E'
                elif name in n_str: day_map[eid] = 'N'
            hist.append(day_map)
        return hist
    except gspread.exceptions.WorksheetNotFound:
        print(f"   ⚠️ Warning: {PREV_SCHED_TAB} not found. Starting clean without history.")
        return []

history = get_history()

# --- 🔍 VERIFICATION PRINTER ---
if history:
    print("\n🔍 --- VERIFYING FETCHED HISTORY ---")
    print("   (Legend: L=Off, M=Morning, E=Evening, N=Night)")
    for i, day_data in enumerate(history):
        formatted_day = " | ".join([f"{reverse_map[eid]}: {shift}" for eid, shift in day_data.items()])
        print(f"   Day -{len(history) - i}: {formatted_day}")
    print("------------------------------------\n")

# ==========================================
# --- 3. DATA & TARGET CALCULATIONS ---
# ==========================================
print("🧮 Calculating targets, holidays, and leave requests...")
raw_data = input_ws.get_all_records()

correct_num_days = calendar.monthrange(TARGET_YEAR, TARGET_MONTH)[1]
raw_data = raw_data[:correct_num_days]
num_days = len(raw_data)

leave_counts = {e: 0 for e in emp_ids}
for row in raw_data:
    for name, eid in emp_map.items():
        if str(row.get(name, '')).upper().strip() == 'L':
            leave_counts[eid] += 1

sundays = sum(1 for week in calendar.monthcalendar(TARGET_YEAR, TARGET_MONTH) if week[calendar.SUNDAY] != 0)
saturdays = sum(1 for week in calendar.monthcalendar(TARGET_YEAR, TARGET_MONTH) if week[calendar.SATURDAY] != 0)

num_holidays = len(PUBLIC_HOLIDAYS)

caps = {}
base_targets = {}

for name, data in EMPLOYEES.items():
    eid, role = data['id'], data['role']
    if role == 'System_Engg':
        base = num_days - sundays
    elif role == 'Full_Time':
        base = num_days - (sundays + saturdays)
    else:
        base = num_days

    base_targets[eid] = base
    caps[eid] = base - leave_counts[eid] - num_holidays

# ==========================================
# --- 4. SOLVER SETUP & STRICT RULES ---
# ==========================================
print("🧱 Building Hard Rules & Constraints...")
MAX_CONSECUTIVE_WORK_DAYS = {
    'System_Engg': 6,
    'Full_Time': 5
}

def get_employee_config(employee_id):
    for name, data in EMPLOYEES.items():
        if data['id'] == employee_id:
            return data['role'], MAX_CONSECUTIVE_WORK_DAYS[data['role']]
    return None, None

model = cp_model.CpModel()
work = {}
is_off = {}

for e in emp_ids:
    for d in range(num_days):
        is_off[(e, d)] = model.NewBoolVar(f'off_{e}_{d}')
        for s in range(3):
            work[(e, d, s)] = model.NewBoolVar(f'w_{e}_{d}_{s}')
        model.Add(sum(work[(e, d, s)] for s in range(3)) == 0).OnlyEnforceIf(is_off[(e, d)])
        model.Add(sum(work[(e, d, s)] for s in range(3)) == 1).OnlyEnforceIf(is_off[(e, d)].Not())

# RULE 1: STRICT LEAVE
for d in range(num_days):
    row_data = raw_data[d]
    for name, eid in emp_map.items():
        if str(row_data.get(name, '')).upper().strip() == 'L':
            model.Add(is_off[(eid, d)] == 1)

# RULE 2: SOFT COVERAGE (ALLOW BLANK SPOTS IF IMPOSSIBLE)
missing_shifts = []
for d in range(num_days):
    for s in range(3):
        missing_var = model.NewBoolVar(f'missing_{d}_{s}')
        # Either someone works, OR we trigger the missing_var penalty
        model.Add(sum(work[(e, d, s)] for e in emp_ids) >= 1).OnlyEnforceIf(missing_var.Not())
        missing_shifts.append(missing_var)
    for e in emp_ids:
        model.Add(sum(work[(e, d, s)] for s in range(3)) <= 1)

# RULE 3: SHIFT RESTRICTIONS
for name, data in EMPLOYEES.items():
    eid, allowed = data['id'], data['shifts']
    for d in range(num_days):
        for s in range(3):
            if s not in allowed:
                model.Add(work[(eid, d, s)] == 0)

# RULE 4: HEALTH ROTATION
for e in emp_ids:
    for d in range(num_days - 1):
        model.Add(work[(e, d, 2)] + work[(e, d + 1, 0)] <= 1)
        model.Add(work[(e, d, 1)] + work[(e, d + 1, 0)] <= 1)
        model.Add(work[(e, d, 2)] + work[(e, d + 1, 1)] <= 1)

# RULE 6: SINTO WEEKEND BAN
for d in range(num_days):
    day_of_week = calendar.weekday(TARGET_YEAR, TARGET_MONTH, d + 1)
    if day_of_week in [5, 6]:
        for s in range(3): model.Add(work[('E', d, s)] == 0)

# ==========================================
# --- 5. BRIDGE CONSTRAINTS & STREAK LOGIC ---
# ==========================================
print("🌉 Applying Cross-Month History Bridge & Streak Logic...")
print("🔍 --- STARTING STREAKS FROM PREVIOUS MONTH ---")

for e in emp_ids:
    role, max_consec = get_employee_config(e)
    name = reverse_map[e]

    if history:
        last_day = history[-1][e]
        if last_day == 'N':
            model.Add(work[(e, 0, 0)] == 0)
            model.Add(work[(e, 0, 1)] == 0)
        elif last_day == 'E':
            model.Add(work[(e, 0, 0)] == 0)

    streak = 0
    if history:
        for h in reversed(history):
            if h[e] != 'L': streak += 1
            else: break

    print(f"   {name}: Started with {streak} days continuous work.")

    for d in range(-streak, num_days - max_consec):
        window_work = []
        for i in range(max_consec + 1):
            day_idx = d + i
            if day_idx < 0:
                window_work.append(1)
            else:
                window_work.append(sum(work[(e, day_idx, s)] for s in range(3)))
        model.Add(sum(window_work) <= max_consec)

print("-----------------------------------------------\n")

# ==========================================
# --- 6. OBJECTIVE & TARGET MATCHER ---
# ==========================================
print("🎯 Applying Scoring Weights & Soft Rules...")
objective_terms = []

# MASSIVE penalty for leaving a spot blank (forces AI to do it only if totally impossible)
UNCOVERED_SHIFT_PENALTY = -50000
for m_var in missing_shifts:
    objective_terms.append(m_var * UNCOVERED_SHIFT_PENALTY)

TARGET_PENALTY = -10000
for e in emp_ids:
    if e == 'E': continue
    target = caps[e]
    shifts_worked = sum(work[(e, d, s)] for d in range(num_days) for s in range(3))
    error = model.NewIntVar(0, num_days * 3, f'err_{e}')
    model.Add(error >= shifts_worked - target)
    model.Add(error >= target - shifts_worked)
    objective_terms.append(error * TARGET_PENALTY)

for e in emp_ids:
    for d in range(num_days - 1):
        for s in range(3):
            stay = model.NewBoolVar(f'stay_{e}_{d}_{s}')
            model.AddMultiplicationEquality(stay, [work[(e, d, s)], work[(e, d+1, s)]])
            objective_terms.append(stay * WEIGHTS['STAY_SAME_SHIFT'])
        doff = model.NewBoolVar(f'doff_{e}_{d}')
        model.AddBoolAnd([is_off[(e, d)], is_off[(e, d+1)]]).OnlyEnforceIf(doff)
        objective_terms.append(doff * WEIGHTS['DOUBLE_OFF_REWARD'])
    for d in range(1, num_days - 1):
        sand = model.NewBoolVar(f'sand_{e}_{d}')
        model.AddBoolAnd([is_off[(e, d)], is_off[(e, d-1)].Not(), is_off[(e, d+1)].Not()]).OnlyEnforceIf(sand)
        objective_terms.append(sand * WEIGHTS['SINGLE_OFF_PENALTY'])

main_work = sum(work[(e, d, s)] for e in emp_ids if e != 'E' for d in range(num_days) for s in range(3))
sinto_work = sum(work[('E', d, s)] for d in range(num_days) for s in range(3))
model.Maximize(main_work * WEIGHTS['MAIN_WORK_DAY'] + sinto_work * WEIGHTS['SINTO_WORK_ADJ'] + sum(objective_terms))

# ==========================================
# --- 7. SOLVER CALCULATION ---
# ==========================================
print("🧠 Handing over to Google OR-Tools AI Solver (Max limit: 60 seconds)...")
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 60
status = solver.Solve(model)

# ==========================================
# --- 8. OUTPUT, ALERTS & DIAGNOSTICS ---
# ==========================================
if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
    print("✅ Solution found! Formatting and pushing to Google Sheets...")
    sched_output = [['Date', '7am to 4pm', '3pm to 12am', '11pm to 8am']]
    assigned_counts = {e: 0 for e in emp_ids}

    for d in range(num_days):
        row = [raw_data[d].get('Date', f'Day {d+1}')]
        for s in range(3):
            # This logic captures the blank spot!
            workers = [reverse_map[e] for e in emp_ids if solver.Value(work[(e, d, s)])]

            if not workers:
                row.append("⚠️ UNCOVERED")
            else:
                row.append(", ".join(workers))
                for w in workers:
                    assigned_counts[emp_map[w]] += 1
        sched_output.append(row)

    summary_output = [['Employee', 'Role', 'Base Target', 'Leave (L)', 'Pub Holidays', 'Final Target', 'Assigned']]
    for name, data in EMPLOYEES.items():
        eid, role = data['id'], data['role']
        summary_output.append([
            name,
            role,
            base_targets[eid],
            leave_counts[eid],
            num_holidays,
            caps[eid] if eid != 'E' else 'Wildcard',
            assigned_counts[eid]
        ])

    sched_ws.clear()
    sched_ws.update(values=sched_output, range_name='A1')
    sched_ws.update(values=summary_output, range_name='G1')

    # Advanced Formatting Block
    requests = []
    for r_idx, row in enumerate(sched_output[1:], start=1):
        for c_idx, name_str in enumerate(row[1:], start=1):
            if "UNCOVERED" in name_str:
                # Highlights the blank spot in BRIGHT RED
                requests.append({"updateCells": {"range": {"sheetId": sched_ws.id, "startRowIndex": r_idx, "endRowIndex": r_idx + 1, "startColumnIndex": c_idx, "endColumnIndex": c_idx + 1}, "rows": [{"values": [{"userEnteredFormat": {"backgroundColor": {"red": 1.0, "green": 0.4, "blue": 0.4}, "textFormat": {"bold": True}}}]}], "fields": "userEnteredFormat(backgroundColor,textFormat)"}})
            elif "," in name_str:
                # Highlights a double-staffed shift in YELLOW
                requests.append({"updateCells": {"range": {"sheetId": sched_ws.id, "startRowIndex": r_idx, "endRowIndex": r_idx + 1, "startColumnIndex": c_idx, "endColumnIndex": c_idx + 1}, "rows": [{"values": [{"userEnteredFormat": {"backgroundColor": {"red": 1.0, "green": 0.95, "blue": 0.8}}}]}], "fields": "userEnteredFormat.backgroundColor"}})
            elif name_str in COLOR_MAP:
                # Regular coloring
                requests.append({"updateCells": {"range": {"sheetId": sched_ws.id, "startRowIndex": r_idx, "endRowIndex": r_idx + 1, "startColumnIndex": c_idx, "endColumnIndex": c_idx + 1}, "rows": [{"values": [{"userEnteredFormat": {"backgroundColor": COLOR_MAP[name_str]}}]}], "fields": "userEnteredFormat.backgroundColor"}})

    if requests: sh.batch_update({"requests": requests})
    print(f"🎉 SUCCESS! Target Month {TARGET_MONTH}/{TARGET_YEAR} processed and uploaded to Google Sheets.")

else:
    print("\n❌ INFEASIBLE! The AI could not find a legal schedule.")
    print("\n" + "="*45)
    print("🔍 AUTO-DIAGNOSTIC REPORT")
    print("="*45)
    print("Analyzing your input data for common mathematical conflicts...\n")

    critical_days = 0
    for d in range(num_days):
        leaves_today = sum(1 for name, eid in emp_map.items() if str(raw_data[d].get(name, '')).upper().strip() == 'L')
        if leaves_today >= 2:
            print(f"   ⚠️ CONFLICT DETECTED: On Day {d+1}, {leaves_today} people requested Leave ('L').")
            critical_days += 1

        day_of_week = calendar.weekday(TARGET_YEAR, TARGET_MONTH, d + 1)
        if day_of_week in [5, 6] and leaves_today >= 2:
            print(f"   🚨 CRITICAL WEEKEND FAILURE: Day {d+1} is a Weekend and has {leaves_today} leaves.")

    if critical_days == 0:
        print("   ✅ No overloaded single-day leaves detected.")

    print("\n   ℹ️ TROUBLESHOOTING TIP:")
    print("      If no specific days were flagged above, someone has likely hit their")
    print("      'Max Consecutive Work Days' limit (5 or 6 days).")
    print("      Try reducing someone's target or removing a Leave ('L') request.")
    print("\n" + "="*45)

🚀 Starting AI Scheduling Engine...
📂 Connecting to Google Sheets (Target: 3/2026)...
⏪ Fetching historical schedule data...

🔍 --- VERIFYING FETCHED HISTORY ---
   (Legend: L=Off, M=Morning, E=Evening, N=Night)
   Day -7: Achu: N | Sidharth: L | Rinshad: M | Sanjeev: E | Sinto: L
   Day -6: Achu: L | Sidharth: L | Rinshad: E | Sanjeev: N | Sinto: M
   Day -5: Achu: M | Sidharth: L | Rinshad: E | Sanjeev: N | Sinto: L
   Day -4: Achu: E | Sidharth: M | Rinshad: L | Sanjeev: N | Sinto: L
   Day -3: Achu: E | Sidharth: M | Rinshad: N | Sanjeev: L | Sinto: L
   Day -2: Achu: E | Sidharth: M | Rinshad: N | Sanjeev: L | Sinto: L
   Day -1: Achu: E | Sidharth: M | Rinshad: N | Sanjeev: L | Sinto: L
------------------------------------

🧮 Calculating targets, holidays, and leave requests...
🧱 Building Hard Rules & Constraints...
🌉 Applying Cross-Month History Bridge & Streak Logic...
🔍 --- STARTING STREAKS FROM PREVIOUS MONTH ---
   Achu: Started with 5 days continuous work.
   Sidharth: Start


---

### 🧱 I. HARD RULES (The Unbreakable Laws)

If the AI cannot satisfy every single one of these simultaneously, it throws its hands up and returns `INFEASIBLE!`.

1. **Strict Leave Guarantee:** If an employee has an `L` on the input sheet for a specific day, they are 100% guaranteed not to be scheduled.
2. **Basic Coverage Rules:**
* Every shift (Morning, Evening, Night) must have **exactly 1** person working.
* Nobody is allowed to work more than **1 shift per 24-hour day** (No double shifts).


3. **Shift Type Restrictions:** Employees can only work the shifts assigned to them in the configuration.
* *Example:* Rinshad is banned from Evenings. Sinto is Morning only.


4. **Health Rotation (No Quick Returns):** The AI enforces "forward rotation" to ensure proper sleep between shifts. It strictly bans:
* ❌ Night shift → Morning shift
* ❌ Evening shift → Morning shift
* ❌ Night shift → Evening shift


5. **Role-Based Burnout Limits (The Smart Rule):**
* **System Engg (Achu):** Can work a maximum of **6 days** in a row.
* **Full Time (Everyone else):** Can work a maximum of **5 days** in a row.


6. **Anti-Hoarding Rest Days:** An employee cannot be scheduled for 4 or more days off in a row (capped at 3), *unless* they specifically requested those days as `L`eave.
7. **Weekend Ban for Sinto:** Sinto is completely locked out of working on Saturdays and Sundays.
8. **Cross-Month History Bridge:** The AI remembers the last week of the previous month. It prevents someone from carrying over a 5-day work streak into the new month without a break, and it prevents illegal turnarounds (like working a Night shift on Feb 28th and a Morning shift on March 1st).

---

### 🎯 II. SOFT RULES (The Scoring System)

These rules don't cause crashes. Instead, the AI uses a point system to weigh its choices. It tries to maximize its total score.

1. **The Target Matcher (-10,000 Points):** This is the ultimate priority. If any main team member misses their exact target calculation (`Base Target - Leaves`), the AI takes a devastating 10,000-point penalty. It will do almost anything to hit this number perfectly.
2. **Keep the Wildcard Benched (-700 Points):** The AI takes a massive 700-point penalty every time it gives a shift to Sinto. It will completely starve him of shifts unless a Hard Rule forces it to use him.
3. **Main Team Priority (+400 Points):** The AI gets a huge reward every time it assigns a standard shift to Achu, Sidharth, Rinshad, or Sanjeev.
4. **Shift Continuity (+100 Points):** The AI gets a solid reward for keeping an employee on the exact same shift type two days in a row (e.g., Morning → Morning), reducing how often they bounce between time zones.
5. **Double Rest Days (+60 Points):** The AI gets a medium reward if it manages to pair rest days together, giving an employee a full 48 hours off.
6. **Single Day Off Penalty (-50 Points):** The AI takes a penalty if it gives an employee an isolated single day off (a "work-off-work" sandwich). It will try to avoid doing this unless absolutely necessary.

